# Build/update private Kaggle input — AF2 spectral
Jalankan sekali di Colab akun yang dapat mengakses folder `Coffee_Bean_Detection` di Drive. Notebook ini membuat versi baru Dataset Kaggle private yang memuat arsip Faruq-v3, D0 seed 42/123/2026, dan manifest SHA eksplisit. Tidak melatih atau membuka test.

In [ ]:
from google.colab import drive, userdata
drive.mount('/content/drive')
import json, os, shutil, subprocess, sys
from pathlib import Path
REPO=Path('/content/coffee-bean-detection'); BRANCH='agent/af2-spectral-factorization'
os.chdir('/content')
if REPO.exists(): shutil.rmtree(REPO)
for attempt in range(3):
    result=subprocess.run(['git','clone','--depth','1','--branch',BRANCH,'https://github.com/ediprin/coffee-bean-detection.git',str(REPO)])
    if result.returncode==0: break
    if REPO.exists(): shutil.rmtree(REPO)
else: raise RuntimeError('git clone gagal tiga kali')
subprocess.run([sys.executable,'-m','pip','install','-q','-e',str(REPO),'kaggle'],check=True)
sys.path.insert(0,str(REPO/'src')); os.chdir(REPO)
from coffee_detector.drive_project import resolve_drive_project_root
from coffee_detector.experiments.prepare_af2_spectral_kaggle import build_af2_spectral_kaggle_bundle
PROJECT_ROOT=resolve_drive_project_root()
BUNDLE=Path('/content/af2-spectral-kaggle-core-v1')
if BUNDLE.exists(): shutil.rmtree(BUNDLE)
manifest=build_af2_spectral_kaggle_bundle(PROJECT_ROOT,BUNDLE)
assert manifest['test_images_included'] is False
print('PROJECT:',PROJECT_ROOT); print(json.dumps(manifest,indent=2))

In [ ]:
# Atur Colab secrets KAGGLE_USERNAME dan KAGGLE_KEY (keduanya plain text).
username=userdata.get('KAGGLE_USERNAME'); key=userdata.get('KAGGLE_KEY')
assert username and key, 'Tambahkan Colab secrets KAGGLE_USERNAME dan KAGGLE_KEY lalu aktifkan notebook access.'
os.environ['KAGGLE_USERNAME']=username; os.environ['KAGGLE_KEY']=key
dataset_id=f'{username}/faruq-v3-experiment-core-v1'
metadata={'title':'Faruq V3 Experiment Core V1','id':dataset_id,'licenses':[{'name':'other'}],'isPrivate':True}
(BUNDLE/'dataset-metadata.json').write_text(json.dumps(metadata,indent=2))
# Dataset sudah ada: membuat VERSION baru. Bila belum ada, ubah `version` menjadi `create`.
command=['kaggle','datasets','version','-p',str(BUNDLE),'-m','Add AF2 spectral manifest and seed-matched D0 checkpoints']
subprocess.run(command,check=True)
print('SELESAI:',f'https://www.kaggle.com/datasets/{dataset_id}')
print('Di notebook Kaggle: buka Input > refresh, lalu pastikan versi terbaru dataset ini terpasang.')